In [ ]:
%pip install pandas
%pip install openpyxl
%pip install geopy
%pip install requests
%pip install -e ../../../

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile

from water_futures_battle.core.base_model import StaticProperties
from utils import upload_csv_files_glob, reconstruct_CBS_fullname

In [ ]:
# Unzip the folder '70739ned-202502190000'
if not Path("../raw-data/70739ned-202502190000").exists():
    with zipfile.ZipFile('../raw-data/70739ned-202502190000.zip', 'r') as zip_ref:
        zip_ref.extractall('../raw-data')

# Upload the file 'RegioSCodes.csv' from that folder
cbs_df = pd.read_csv('../raw-data/70739ned-202502190000/RegioSCodes.csv', delimiter=';',dtype=str)

jurisdiction_sp = StaticProperties('jurisdictions-static_properties')

for sheet, id_begin in zip(['regions', 'provinces', 'municipalities'], ['LD', 'PV', 'GM']):
    belongs_to_sheet = cbs_df['Identifier'].str.startswith(id_begin)
    # Create the dataframe for this sheet with name (Title) and cbs_id (Identifier) only
    sheet_df = cbs_df.loc[belongs_to_sheet, ['Title', 'Identifier']].copy()
    sheet_df.columns = ['name', 'cbs_id']
    # Add the dataframe global properties:
    # - units dimensionless for every column
    # - pattern false
    # - type: str for every column
        # ...existing code...
    sheet_df.attrs['units'] = 'dimensionless'
    sheet_df.attrs['pattern'] = False
    sheet_df.attrs['type'] = 'str'
    jurisdiction_sp[sheet] = sheet_df

# All regions are in the Netherlands ficticiou cbs_id NL0000
jurisdiction_sp['regions']['state'] = 'NL0000'

In [ ]:
# Which province belongs to which region is written in the RegioSCode Description for the regions
for i in range (1, 5):
    print(cbs_df[cbs_df['Identifier']==f'LD0{i}']['Description'].iloc[0])

In [ ]:
province_to_region = {
    # Noord-Nederland
    'PV20': 'LD01', # Groningen
    'PV21': 'LD01', # Fryslan
    'PV22': 'LD01', # Drenthe
    # Oost-Nederland
    'PV23': 'LD02', # Overijssel
    'PV24': 'LD02', # Gelderland
    'PV25': 'LD02', # Flevoland
    # West-Nederland
    'PV27': 'LD03', # Noord-Holland
    'PV28': 'LD03', # Zuid-Holland
    'PV26': 'LD03', # Utrecht
    'PV29': 'LD04', # Zeeland
    # Zuid-Nederland
    'PV30': 'LD04', # Noord-Brabant
    'PV31': 'LD04', # Limburg
}

# Add the 'region' column to the provinces DataFrame
provinces_df = jurisdiction_sp['provinces']
provinces_df['region'] = provinces_df['cbs_id'].map(province_to_region)

#fix Fryslan name without special character
provinces_df.loc[provinces_df['cbs_id']=='PV21', 'name'] = 'Fryslan'

In [ ]:
# While to map the municipalities to the provinces, we can use the DimensionGroupId of the table
municipalities_df = jurisdiction_sp['municipalities']
# Map municipality cbs_id to province cbs_id using cbs_df['DimensionGroupId']
municipality_to_province = (
    cbs_df.set_index('Identifier')['DimensionGroupId']
    .str[2:]  # Remove the first two characters (e.g., 'GM' from 'GMPV30')
    .to_dict()
    # This maps municipality cbs_id (Identifier) to province cbs_id (e.g., PV30)
    # Only for municipalities, so filter keys that start with 'GM'
)
municipalities_df['province'] = municipalities_df['cbs_id'].map(municipality_to_province)

# Remove the two extra fake Municipalities "Buitenland" and "Niet-gemeentelijk ingedeeld"
municipalities_df.drop(municipalities_df[municipalities_df['cbs_id'].isin(['GM0998', 'GM0999'])].index, inplace=True)

The other properties that we have are:
- begin_date 
- end_date

These are obtained from a different file called Observations.csv (delimiter ';').
The code R00005 and R00006 are for these two properties and the suffix _1 or _2 indicates if the date is in the format  01 Jan 2015 or 20150101.
For each Municipality, we extract its R0005_2 and R0006_2
Parse it to generate the date and add it to a the municipalites_dynamic_property file 
then we will drop any municipalities that doesn't exist before 2000!

In [ ]:
# Upload the file
observations_df = pd.read_csv('../raw-data/70739ned-202502190000/Observations.csv', delimiter=';',dtype=str)
municipalities_bd = pd.DataFrame(columns=municipalities_df['cbs_id'].tolist(), index=pd.Index([], name='timestamp'))
municipalities_ed =  pd.DataFrame(columns=municipalities_df['cbs_id'].tolist(), index=pd.Index([], name='timestamp'))
    
begin_dates_masks = observations_df['Measure'] == 'R00005_2'
end_dates_masks = observations_df['Measure'] == 'R00006_2'
for idx, municipality_static_p in municipalities_df.iterrows():
    muni_id = municipality_static_p['cbs_id']
    # Get the date in 20150101 format, then add the dashed using the chars position
    value_date:str = observations_df['StringValue'][begin_dates_masks & (observations_df['RegioS']==muni_id)].iloc[0]
    dashed_date = value_date[0:4]+'-'+value_date[4:6]+'-'+value_date[6:8]
    
    # Ensure the row exists, if not, create it with all None
    if dashed_date not in municipalities_bd.index:
        municipalities_bd.loc[dashed_date] = [False] * len(municipalities_bd.columns)
    
    municipalities_bd.at[dashed_date, muni_id] = True

    # Same stuff for the end date, but some end dates will be missing!
    value_date:str = observations_df['StringValue'][end_dates_masks & (observations_df['RegioS']==muni_id)].iloc[0]
    if value_date is not None and not pd.isna(value_date):
        dashed_date = value_date[0:4]+'-'+value_date[4:6]+'-'+value_date[6:8]
        if dashed_date not in municipalities_ed.index:
            municipalities_ed.loc[dashed_date] = [False] * len(municipalities_ed.columns)
        municipalities_ed.at[dashed_date, muni_id] = True

municipalities_bd.sort_index(inplace=True)
municipalities_ed.sort_index(inplace=True)

In [ ]:
# Now drop from the static properties and the dynamic ones all municipalities that don't exist anymore from 2000-01-01
ed_dates = pd.to_datetime(municipalities_ed.index, errors='coerce')
pre2000_df = municipalities_ed[ed_dates <= pd.Timestamp('2000-01-01')]
superold_munis = pre2000_df.columns[pre2000_df.any()].tolist()

# Let's save for backup the current status of the municipalities one excel file with 3 sheets
# We can use it to check if we deleted municipalities that we should have not
with pd.ExcelWriter(f"../.temp/municipalities-all-cbs_processed_info.xlsx") as writer:
    for sheet, sheet_df in zip(['cbs_base_info', 'begin_date', 'end_date'], [municipalities_df, municipalities_bd, municipalities_ed]):
        sheet_df.to_excel(writer, sheet_name=sheet)

# Now actually remove them, and then also drop all rows full of False (useless dates)
municipalities_df.drop(municipalities_df[municipalities_df['cbs_id'].isin(superold_munis)].index, inplace=True)
for df in [municipalities_bd, municipalities_ed]:
    df.drop(columns=superold_munis, inplace=True)
    df.drop(df[(df == False).all(axis=1)].index, inplace=True)


In [ ]:
# We need to parse, somehow the information telling us where the closed municipalities went
import re
from typing import List, Tuple, Dict, Any

def parse_description_indexed(description_text: str, source_cbs_id: str) -> Tuple[Tuple[str,List[str]], List[str]]:
    """
    Parses the full description text line-by-line using a line index to extract 
    destination municipalities from 'Opgeheven per' and 'Naamswijziging per' 
    events, as they explain why the municipality code (source_cbs_id) was lifted
    and where

    Args:
        description_text (str): The raw 'Description' string (including newlines).
        source_cbs_id (str): The 'cbs_id' of the municipality the description belongs to.

    Returns:
        tuple: (list of results, list of log messages)
            - results: List of tuples (event_type, cbs_code, name, date_str).
            - log: List of strings containing parsing actions and outcomes.
    """
    
    # State and Data initialization
    ed_reason: str = 'Unknown'
    ed_cbsids: List[str] = []
    log_messages: List[str] = [f"--- Starting Parse for {source_cbs_id} ---"]
    
    # Define Regex Patterns (compiled for efficiency)
    opgeheven_header_pattern = re.compile(r'Opgeheven per (\d{2}-\d{2}-\d{4}), overgegaan naar:')
    naamswijziging_header_pattern = re.compile(r'Naamswijziging per (\d{2}-\d{2}-\d{4})')
    naamswijziging_pattern = re.compile(r'nieuwe naam:\s*([^,]+?)\s*\((GM\d{4})\)')
    municipality_line_pattern = re.compile(r'^\s*-\s*([^,]+?)\s*\((GM\d{4})\)')
    
    lines = description_text.splitlines()
    num_lines = len(lines)
    line_index = 0

    def close():
        log_messages.append(f"--- Parsing Finished ---")
    
        return (ed_reason, ed_cbsids), log_messages
    
    while line_index < num_lines:
        current_line = lines[line_index].strip()
        
        if not current_line:
            line_index += 1
            continue

        # --- Case 1: Opgeheven per (Merge/Integration) ---
        if current_line.startswith('Opgeheven per'):
            header_match = opgeheven_header_pattern.search(current_line)
            if header_match:
                date_str = header_match.group(1)
                log_messages.append(f"  > Detected **Opgeheven per** event (Date: {date_str}). Starting list scan.")
                ed_reason = 'lifted'

                # Move index to the next line to start list parsing
                line_index += 1
                
                # Start loop for list items
                while line_index < num_lines:
                    list_line = lines[line_index].strip()
                    
                    # Check if the line is a list item starting with "-"
                    municipality_match = municipality_line_pattern.search(list_line)
                    
                    if municipality_match:
                        # Success: Found a destination municipality
                        name = municipality_match.group(1).strip()
                        cbs_code = municipality_match.group(2)
                        
                        ed_cbsids.append(cbs_code)
                        log_messages.append(f"  ✅ Extracted destination: {cbs_code} ({name})")
                        line_index += 1 # Move to the next line in the list
                    # Follow up lines with Gemeente properties (land, wonings, people)
                    elif list_line.startswith('...'): 
                        line_index += 1 # Skip this line

                    else:
                        # Failure: Line is not a list item, so the block ends.
                        log_messages.append(f"  > Block ended at line {line_index + 1}. Moving to next main event.")
                        break # Exit the inner while loop
                
                return close()

        # --- Case 2: Naamswijziging per (Renaming) ---
        elif current_line.startswith('Naamswijziging per'):
            header_match = naamswijziging_header_pattern.search(current_line)
            date_str = header_match.group(1)
            log_messages.append(f"  > Detected **Naamswijziging per** event (Date: {date_str}).")
            ed_reason = 'renamed'

            line_index += 1
            current_line = lines[line_index].strip()

            match = naamswijziging_pattern.search(current_line)
            if match:
                name = match.group(1).strip()
                cbs_code = match.group(2)
                
                ed_cbsids.append(cbs_code)
                log_messages.append(f"  🔄 Extracted new name: {name} ({cbs_code})")
            else:
                log_messages.append(f"  ⚠️ Could not parse Naamswijziging on line {line_index}.")
            
            
            return close()
            
        # --- Case 3: Other Headers (Ignore and Quit) ---
        elif any(current_line.startswith(h) for h in ['Ontstaan per', 'Wijziging per', 'Grenswijziging per', 'Begindatum']):
            log_messages.append(f"  --- Found Ignored Header: **{current_line.split(' ', 1)[0]}**. Quitting scan. ---")
            break # Exit the main while loop

        # --- Default Case: Move to Next Line ---
        line_index += 1

    return close()

closed_cbs_ids = municipalities_ed.columns[municipalities_ed.any(axis=0)]
descriptions_closed_munis = cbs_df[cbs_df['Identifier'].isin(closed_cbs_ids)]['Description']

# Turn the dynamic objets into object types so that I can set them to strings too
municipalities_ereason = municipalities_ed.copy().astype(object)
municipalities_ereason[municipalities_ed == False] = None
municipalities_ecbsids = municipalities_ed.copy().astype(object)
municipalities_ecbsids[municipalities_ed == False] = None

for descritpion_closed_muni, closed_cbs_id in zip(descriptions_closed_munis, closed_cbs_ids):
    (ed_reason, ed_cbsids), log = parse_description_indexed(descritpion_closed_muni, closed_cbs_id)
    mask_date = municipalities_ed[closed_cbs_id].values
    municipalities_ereason.loc[mask_date, closed_cbs_id] = ed_reason
    municipalities_ecbsids.loc[mask_date, closed_cbs_id] = ';'.join(ed_cbsids)
    
    """
    print("\n" + "=" * 50)
    print(f"RESULTS SUMMARY for {closed_cbs_id}:")
    print(f"Reason: {ed_reason}")
    for cbs_code in ed_cbsids:
        print(f"- {cbs_code}")
    print("=" * 50)
    
    print("\n" + "#" * 50)
    print("PARSING LOG FILE (Line-by-Line Trace):")
    print("#" * 50)
    for msg in log:
        print(msg)
    print("#" * 50)
    """

In [ ]:
# Now from the big tables, compress the information into municipalites_df

# Add the columns 'begin_date', 'end_date', 'end_reason', 'destination_cbs_ids' with all None (type object)
for col in ['begin_date', 'end_date', 'end_reason', 'destination_cbs_ids']:
    municipalities_df[col] = None
    municipalities_df[col] = municipalities_df[col].astype(object)  # Only set these columns to object

# Finally for all dates above 2000-01-01 that are not on 1st of January, convert to first of January of the following year
# so that we are aligned with the data on the CBS
def to_next_jan_first(dt: str | None) -> str | None:
    if not dt or dt == '':
        return ''
    year, month, day = dt.split('-')
    if month == "01" and day == "01":
        return dt
    # Move to Jan 1 of next year
    nY = str(int(year)+1)
    return f"{nY}-01-01"

# For each municipality, extract the relevant values from the dynamic tables
for idx, row in municipalities_df.iterrows():
    muni_id = row['cbs_id']
    # BEGIN DATE
    begin_mask = municipalities_bd[muni_id] == True
    if begin_mask.any():
        municipalities_df.at[idx, 'begin_date'] = to_next_jan_first(municipalities_bd.index[begin_mask][0])
    # END DATE
    end_mask = municipalities_ed[muni_id] == True
    if end_mask.any():
        municipalities_df.at[idx, 'end_date'] = to_next_jan_first(municipalities_ed.index[end_mask][0])
    # END REASON
    if end_mask.any():
        municipalities_df.at[idx, 'end_reason'] = municipalities_ereason[muni_id][end_mask].iloc[0]
    # END CBS IDS
    if end_mask.any():
        municipalities_df.at[idx, 'destination_cbs_ids'] = municipalities_ecbsids[muni_id][end_mask].iloc[0]

We discovered that:
- Municipality Loosdrecht (GM0330) closes to Wijdemeren (GM1696), which is in a different province (Noord-Holland (PV27)) than the original (Utrecht (PV26))
- Municipality Leerdam (GM0545) closes to Vijfheerenlanden (GM1961), which is in a different province (Utrecht (PV26)) than the original (Zuid-Holland (PV28))
- Municipality Zederik (GM0707) closes to Vijfheerenlanden (GM1961), which is in a different province (Utrecht (PV26)) than the original (Zuid-Holland (PV28))

Therefore, we modify these municipalities to be already in the destination province.

In [ ]:
municipalities_df.loc[municipalities_df['cbs_id']=='GM0330', 'province'] = 'PV27'
municipalities_df.loc[municipalities_df['cbs_id']=='GM0545', 'province'] = 'PV26'
municipalities_df.loc[municipalities_df['cbs_id']=='GM0707', 'province'] = 'PV26'

In [ ]:
# Add the effective cbs_id once and for all. 
def destination_municipality(municipality: pd.Series, lookup_df: pd.DataFrame) -> str:
        """
        Returns the municipality CBS ID that inherits the hydraulic connections for
        a closing municipality
        """
        if municipality['end_reason'] is None or municipality['end_reason'] == '':
            return ''

        destinations = municipality['destination_cbs_ids'].split(';')
        if municipality['end_reason'] == 'renamed':
            assert len(destinations) == 1
            return destinations[0]
        
        # should be lifted
        if len(destinations) == 1:
            return destinations[0]
        
        # lifted, and more than one, we had a complex approach using population and more.
        
        # We get if a municipality is opening the same year
        new_municipalities = []
        for m in destinations:
            if pd.to_datetime(lookup_df.loc[m, 'begin_date']) == pd.to_datetime(municipality['end_date']):
                new_municipalities.append(m)
            
        # If there is only one new municipality, assign there
        if len(new_municipalities) == 1:
            print(f"Municipality {municipality['cbs_id']} to {new_municipalities[0]}")
            return new_municipalities[0]
        
        # More, postpone the problem

        return 'MISSING'

municipalities_df['main_destination_municipality'] = municipalities_df.apply(
    lambda r: destination_municipality(r, municipalities_df.set_index('cbs_id')),
    axis=1
)

display(municipalities_df[municipalities_df['main_destination_municipality'] == 'MISSING'])

Manual adjustments are inevitable and for missing values too

In [ ]:
manual_adjust_dest_map = {
    # Goes to a new municipality, when it should not.
    'GM0055' : 'GM0080', 
    'GM0194' : 'GM0181',
    'GM0260' : 'GM0240',
    'GM0140' : 'GM1900',
    'GM0292' : 'GM0222',
    # Impossible to fit into a rule.
    'GM0227' : 'GM0258',
    'GM0239' : 'GM0262',
    'GM0788' : 'GM0824',
    'GM0248' : 'GM1876',
    'GM0191' : 'GM0166',
    'GM0548' : 'GM1916',
    'GM0552' : 'GM1783',
    'GM1671' : 'GM0796',
    'GM0555' : 'GM1842',
    'GM0993' : 'GM1507',
    'GM0170' : 'GM0148',
    'GM0278' : 'GM1859',
    'GM0607' : 'GM1842',
    'GM0286' : 'GM1876',
    'GM0188' : 'GM0149',
    'GM0053' : 'GM1966',
    'GM0295' : 'GM1509',
    'GM0466' : 'GM0420',
    'GM0298' : 'GM1876'
}
for cbs_id, destination_municipality in manual_adjust_dest_map.items():
    mask = municipalities_df['cbs_id'] == cbs_id
    municipalities_df.loc[mask, 'main_destination_municipality'] = destination_municipality

In [ ]:
import requests
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12801)

RUN_COORD_API = False  # Set to False to not run the API and use backup data (if available, otherwise defaults to mock-up)

if not RUN_COORD_API:
    try:
        coord_backup_df = pd.read_excel('../.backups/jurisdictions-static_properties-backup-2026-02-19.xlsx', dtype=str, sheet_name='municipalities')
        coord_backup_df = coord_backup_df.set_index('cbs_id')
    except Exception as e:
        print(f"Backup coordinate file could not be loaded: {e}")
        # Create a fake backup with random values for all municipality names in the current df
        coord_backup_df = pd.DataFrame({
            'name': municipalities_df['name'],
            'cbs_id': municipalities_df['cbs_id'],
            'latitude': RNG.uniform(50.75, 53.5, len(municipalities_df)),
            'longitude': RNG.uniform(3.36, 7.22, len(municipalities_df))
        })
        coord_backup_df = coord_backup_df.set_index('cbs_id')

else:
    geolocator = Nominatim(user_agent="netherlands_municipality_finder")

def get_lat_lon(municipality_name, municipality_cbs_id, RUN_API):
    """Geocodes a municipality name to get latitude and longitude."""
    if not RUN_API:
        lat = float(coord_backup_df.loc[municipality_cbs_id, 'latitude'])
        lon = float(coord_backup_df.loc[municipality_cbs_id, 'longitude'])
        return lat, lon

    # Append 'Netherlands' to improve accuracy and limit search scope
    query = f"{municipality_name}, Netherlands"
    try:
        # Use a timeout in case of slow API response
        location = geolocator.geocode(query, timeout=10)
        if location:
            # Add a small delay between requests to avoid hitting rate limits
            time.sleep(1.5) 
            return location.latitude, location.longitude
        else:
            print(f"Could not find coordinates for: {municipality_name}")
            return None, None
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"Error geocoding {municipality_name}: {e}")
        time.sleep(5) # Wait longer after an error
        return None, None

# Apply the function to the 'name' column of all municipalities that do not have
# 'renamed' as reason. For those, we will use the same coordinates of the new one.

mask = municipalities_df['end_reason'] != 'renamed'
new_coords = municipalities_df.loc[mask, ['name', 'cbs_id']].apply(lambda row: get_lat_lon(row['name'], row['cbs_id'], RUN_COORD_API), axis=1)
new_coords_df = pd.DataFrame(new_coords.tolist(), index=new_coords.index, columns=['latitude', 'longitude'])
municipalities_df.loc[mask, ['latitude', 'longitude']] = new_coords_df

In [ ]:
lookup_df = municipalities_df.set_index('cbs_id')

# Now go to the renamed and make sure that they only have one destination municipality (the new name)
mask = (lookup_df['end_reason'] == 'renamed') & (~lookup_df['destination_cbs_ids'].astype(str).str.contains(';'))
renamed_ids = lookup_df.index[mask].astype(str)
for rid in renamed_ids:
    rid_mask = municipalities_df['cbs_id'] == rid
    did = lookup_df.loc[rid, 'destination_cbs_ids']
    municipalities_df.loc[rid_mask, 'latitude'] = lookup_df.loc[did, 'latitude']
    municipalities_df.loc[rid_mask, 'longitude'] = lookup_df.loc[did, 'longitude']

In [ ]:
RUN_ELEV_API = False
elevation_backup_df = pd.read_excel('../.backups/jurisdictions-static_properties-backup-2026-02-19.xlsx', dtype=str, sheet_name='municipalities')
elevation_backup_df = elevation_backup_df.set_index('cbs_id')

if RUN_ELEV_API:
    USE_BACKUP = False
    ELEVATION_API_URL = "https://api.open-elevation.com/api/v1/lookup"
    DECIMAL_PLACES = 5 # Round coordinates to 5 decimal places for matching
    """
    Queries Open-Elevation API using a single POST request and matches results 
    using closest point (Euclidean distance) for API, or exact cbs_id for backup.
    """

    valid_coords = municipalities_df.dropna(subset=['latitude', 'longitude']).copy()
    assert not valid_coords.empty
            
    try:
        # Construct the JSON body and execute the single POST request
        payload = {'locations': valid_coords[['latitude', 'longitude']].to_dict(orient='records')}
    
        response = requests.post(ELEVATION_API_URL, json=payload, timeout=60) 
        response.raise_for_status() 
        data = response.json()
        results = data.get('results', [])

        # 3. For each returned elevation, assign to the closest municipality by Euclidean distance
        api_coords = np.array([[r['latitude'], r['longitude']] for r in results])
        api_elevs = np.array([r['elevation'] for r in results])
        df_coords = valid_coords[['latitude', 'longitude']].to_numpy()
        assigned_elevs = np.empty(len(valid_coords))
        for i, (lat, lon) in enumerate(df_coords):
            dists = np.sqrt((api_coords[:,0] - lat)**2 + (api_coords[:,1] - lon)**2)
            idx = np.argmin(dists)
            assigned_elevs[i] = api_elevs[idx]
        municipalities_df.loc[valid_coords.index, 'elevation'] = assigned_elevs

    except requests.exceptions.RequestException as e:
        print(f"🛑 Error querying Open-Elevation API with single POST request: {e}")
        USE_BACKUP = True

else:
    USE_BACKUP = True

if USE_BACKUP:
    municipalities_df['elevation'] = municipalities_df['cbs_id'].map(
        lambda x: elevation_backup_df.loc[x, 'elevation']
    )


### Manual fixes of elevations

A few municipalities are impossible to feed because too high and without any own source. So to get water from the others we need them to be at the same elevation.

In [ ]:
# Right bottom corner of Limburg 
for municipality_id, z in zip(
    ['GM0981', 'GM1729', 'GM0936', 'GM0928', 'GM0882'],
    [151, 149, 146, 119, 121]
):
    mask = municipalities_df['cbs_id'] == municipality_id
    municipalities_df.loc[mask, 'elevation'] = z

# in Utrecht, Utrechtse Heuvelrug (GM1581) is a merge of 4 municipalities between 6 and 13m.
# for some reason the APIs got 38, way off..
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM1581',
    'elevation'
] = 9 # actual mean of the 4 closing municipalities

# Gelderland
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0233',
    'elevation'
] = 22
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0302',
    'elevation'
] = 25
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0294',
    'elevation'
] = 23
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0241',
    'elevation'
] = 13
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0282',
    'elevation'
] = 13
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0207',
    'elevation'
] = 16

# Zuid holland
# for some reason Rotterdam is at a weird +8 m, which is 10m higher than the area, we will never get the water theere like this
municipalities_df.loc[
    municipalities_df['cbs_id']=='GM0599',
    'elevation'
] = 2

In [ ]:
# Add the touristic info.
# We download them by year to make sure they didn't change but we will assume the constant (aka Static properties)

# Extract from zip:
if not Path('../raw-data/CBS-regionale_kerncijfers_nederland').exists():
    with zipfile.ZipFile('../raw-data/CBS-regionale_kerncijfers_nederland.zip', 'r') as zip_ref:
        zip_ref.extractall('../raw-data')

TINFO_COLUMNS = ['CBS_fullname', 'year',
                 'touristic_area-cbs_id', 'touristic_area-name',
                 'touristic_group-cbs_id', 'touristic_group-name']
TINFO_DTYPES = [
    str, int
] + [str]*4
munis_tinfo_df = upload_csv_files_glob(
    "CBS-regionale_kerncijfers_nederland/CBS-municipalities-tourism_info",
    columns_names=TINFO_COLUMNS,
    columns_dtypes=TINFO_DTYPES
    )

# Let's make sure that for each muni only one code is associated, ignoring nan, '.', and only considering codes starting with 'T'
def filter_valid_touristic_codes(arr):
    # Remove nan, '.', and only keep codes starting with 'T'
    arr = [x for x in arr if isinstance(x, str) and x.startswith('T')]
    return arr

for col in ['touristic_area-cbs_id', 'touristic_group-cbs_id']:
    grouped = munis_tinfo_df.groupby('CBS_fullname')[col].apply(lambda x: filter_valid_touristic_codes(x.unique()))
    for muni, unique_vals in grouped.items():
        if len(unique_vals) > 1:
            print(f"Municipality {muni} has multiple {col} values: {unique_vals}")

In [ ]:
# So, we will use only the data in 2024 anyway, but we know that for Drecherland and Schagen the ids change.
# Let's create a lookup df
munis_lookup = municipalities_df.copy()
munis_lookup['CBS_fullname'] = munis_lookup.apply(lambda x: reconstruct_CBS_fullname(x, cbs_df=munis_tinfo_df, expected_matches=25), axis=1)
munis_lookup.set_index('cbs_id', inplace=True)

munis_tinfo_2024 = munis_tinfo_df[munis_tinfo_df['year'].astype(int) == 2024]
munis_tinfo_2024.set_index('CBS_fullname', inplace=True)

municipalities_df['touristic_area_cbs_id'] = municipalities_df['cbs_id'].apply(
    lambda x: munis_tinfo_2024.loc[munis_lookup.loc[x, 'CBS_fullname'], 'touristic_area-cbs_id']
)
municipalities_df['touristic_group_cbs_id'] = municipalities_df['cbs_id'].apply(
    lambda x: munis_tinfo_2024.loc[munis_lookup.loc[x, 'CBS_fullname'], 'touristic_group-cbs_id']
)

In [ ]:
# Clean out the names before saving: remove the () and strip spaces at the end
for sheet, sheet_df in jurisdiction_sp.items():
    sheet_df['name'] = sheet_df['name'].map(lambda x: x.split('(')[0].strip())

In [ ]:
jurisdiction_sp.dump(path='..')